In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

c:\Users\NIKHIL\MLOPS\mlops_atlas\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
294,"If I had never seen the first Road House, then...",negative
986,"The whole point of making this film, one of th...",positive
113,"""This Man's Navy"" is, as other comments have i...",positive
567,"Still haven't read a single Dan Brown book, bu...",positive
762,Early 80's creature feature concerns a long ab...,positive


In [5]:
# data preprocessing
def lemmitization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """ remove stop words from the text."""
    stops_words = set(stopwords.words("english"))
    text = [word for word in text.split() if word not in stops_words]
    return " ".join(text)

def remove_numbers(text):
    """remove the words for the text"""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def remove_punctuation(text):
    """remove the punctuation from the text"""
    text = re.sub('[%s]' % string.punctuation, '', text)
    text = text.replace(',',"")
    text = re.sub('\s+', ' ', text)
    text = text.strip()
    return text

def removing_url(text):
    """remove the url from the text"""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """normalize the text"""
    try:
        df['review'] = df['review'].apply(lambda x: x.lower())
        df['review'] = df['review'].apply(removing_url)
        df['review'] = df['review'].apply(remove_punctuation)
        df['review'] = df['review'].apply(remove_numbers)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(lemmitization)

        return df

    except Exception as e:
        print(f"Error in normalize_text: {e}")
        raise e

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
294,never seen first road house guess movie might ...,negative
986,whole point making film one earliest best inte...,positive
113,man navy comment indicated rare wellfilmed loo...,positive
567,still havent read single dan brown book watche...,positive
762,early creature feature concern long abandoned ...,positive


In [8]:
df['sentiment'].value_counts()

sentiment
negative    254
positive    246
Name: count, dtype: int64

In [9]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
294,never seen first road house guess movie might ...,0
986,whole point making film one earliest best inte...,1
113,man navy comment indicated rare wellfilmed loo...,1
567,still havent read single dan brown book watche...,1
762,early creature feature concern long abandoned ...,1


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
vectorizer = CountVectorizer(max_features=100)
x = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)